# Outcome Variables female- and foreign skill share

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import statsmodels.formula.api as smf
from rdrobust import rdrobust
from rdrobust import rdbwselect   # needed for the bandwidth-selection step
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col # needed for the regression table output

In [2]:
# Load the cleaned RDD sample built in the data-prep notebook
df = pd.read_csv('df_rdd_clean.csv', dtype={'code_commune': str})
df['code_commune'] = df['code_commune'].str.zfill(5)   # re-enforce zero-padding after CSV round-trip

# 2008 has zero treated units (far-right never won a decisive round that year),
# so the cutoff falls outside empirical support and rdrobust/rdbwselect can't be estimated
df = df[df['election_year'] != 2008].copy()
print(df['election_year'].value_counts())

election_year
2014    513
2020    285
Name: count, dtype: int64


## Bandwidth selection

In [3]:
# MSE-optimal bandwidth per cycle, based on the MAIN outcome (delta_skill_share).
# This h is then frozen and reused for every spec/outcome in this notebook, so that
# balance checks, baseline RDD, and the new gendered outcomes all use the same window
# -> results stay comparable across specifications.
bw_by_cycle = {}
for cycle in [2014, 2020]:
    sub = df[df['election_year'] == cycle].dropna(subset=['delta_skill_share', 'd_50'])
    bw_res = rdbwselect(y=sub['delta_skill_share'], x=sub['d_50'], c=0, bwselect='mserd')
    bw_by_cycle[cycle] = float(bw_res.bws.iloc[0, 0])   # h (left) from the mserd selector
    print(f"{cycle}: h = {bw_by_cycle[cycle]:.4f}")

2014: h = 0.9127
2020: h = 1.7324


## Build new Outcome Variable delta_female_skill_share

In [4]:
# Raw sex-disaggregated education counts (not in df_rdd_clean.csv — need the source file)
df_edu = pd.read_csv('diplomescommunes.csv', dtype={'codecommune': str})
df_edu['codecommune'] = df_edu['codecommune'].str.zfill(5)   # zero-pad to match df's commune codes

def female_skill_share(df, year):
    """Bac-or-above share among women only, for a given census year."""
    educated = df[f'bacf{year}'] + df[f'supf{year}']            # women: bac + university
    return educated / (educated + df[f'nodipf{year}'])          # denominator: women with no diploma added back in

# Same (election_year, pre2_census, pre_census, post_census) windows as delta_skill_share,
# so the female outcome is directly comparable to the baseline one
windows = [(2014, 2004, 2009, 2014), (2020, 2009, 2014, 2019)]

rows = []
for cycle, pre2_y, pre_y, post_y in windows:
    rows.append(pd.DataFrame({
        'commune_id': df_edu['codecommune'],
        'election_year': cycle,
        # pre-treatment placebo change (both years before the election) -> used as the "post=0" row later
        'skill_share_trend_female': female_skill_share(df_edu, pre_y) - female_skill_share(df_edu, pre2_y),
        # actual treatment-period change (straddles the election) -> used as the "post=1" row later
        'delta_female_skill_share': female_skill_share(df_edu, post_y) - female_skill_share(df_edu, pre_y),
    }))
df_female = pd.concat(rows, ignore_index=True)

C:\Users\tobia\AppData\Local\Temp\ipykernel_29952\2766222187.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.


In [5]:
# Merge the female skill share data into the main df
df = df.merge(df_female, left_on=['code_commune', 'election_year'],
              right_on=['commune_id', 'election_year'], 
              how='left' # keep all rows of df even if a female outcome is missing
              ).drop(columns='commune_id') # duplicate key, no longer needed after the merge

print(df['delta_female_skill_share'].isna().sum(), 'missing after merge')  # sanity check

0 missing after merge


## Build Diff-in-Disc estimator (local-linear, triangular kernel, clustered SE)

In [6]:
    # Local-linear Diff-in-Disc: y ~ d_50 * treat * post, restricted to |d_50| <= h,
    # weighted by a triangular kernel (more weight near the cutoff, following rdrobust'sdefault kernel). 
    # treat:post is the Diff-in-Disc estimate — the change in the cutoff-jump from the pre-treatment placebo period to the actual post-election period.
    # SEs are clustered by commune because each commune contributes two rows (pre + post).




def diffindisc(data, h, cluster_col='code_commune'):
    # keep only rows within the bandwidth window
    d = data[data['d_50'].abs() <= h].copy()
    # triangular kernel weight: 1 at the cutoff, shrinking to 0 at the edge of h
    d['w'] = 1 - d['d_50'].abs() / h
    # weighted least squares: y ~ d_50 * treat * post (full interaction, patsy-expanded)
    # cluster-robust SEs by commune (each commune appears twice: pre-row + post-row)
    fit = smf.wls('y ~ d_50 * treat * post', data=d, weights=d['w']).fit(
        cov_type='cluster', cov_kwds={'groups': d[cluster_col]}
    )
    n_communes = d['code_commune'].nunique()
    n_treat = d.loc[d['post'] == 1, 'treat'].sum()
    print(f"h={h:.3f} | communes in window={n_communes} | treated={n_treat}")
    return fit

IndentationError: expected an indented block after function definition on line 9 (3653833040.py, line 11)

In [7]:
line = "    d = data[data['d_50'].abs() <= h].copy()"  # copy the actual first line of your function body here
print([hex(ord(c)) for c in line[:6]])

['0x20', '0x20', '0x20', '0x20', '0x64', '0x20']


In [ ]:
# post=0: pre-treatment placebo change | post=1: actual post-election change
long_base = pd.concat([
    df.assign(post=0, y=df['skill_share_trend']),
    df.assign(post=1, y=df['delta_skill_share']),
])
long_female = pd.concat([
    df.assign(post=0, y=df['skill_share_trend_female']),
    df.assign(post=1, y=df['delta_female_skill_share']),
])

In [ ]:
model_2014_base   = diffindisc(long_base[long_base['election_year'] == 2014],   h=bw_by_cycle[2014])
model_2020_base   = diffindisc(long_base[long_base['election_year'] == 2020],   h=bw_by_cycle[2020])
model_2014_female = diffindisc(long_female[long_female['election_year'] == 2014], h=bw_by_cycle[2014])
model_2020_female = diffindisc(long_female[long_female['election_year'] == 2020], h=bw_by_cycle[2020])

In [ ]:
print(summary_col( # summary_col is a statsmodels function that formats multiple regression results into a single table.
    [model_2014_base, model_2020_base, model_2014_female, model_2020_female],
    model_names=['2014 base', '2020 base', '2014 female', '2020 female'],
    stars=True,
    include_r2=False
))